# Main run on Colab: reference, PINN campaign, evaluation

This notebook runs the whole main experiment of the five-field boundary problem from start to finish and keeps everything in your Google Drive.

**How to use it**

1. *Runtime > Change runtime type*: pick a GPU (T4 or better) if you have one. CPU works too, only slower.
2. Check the settings cell below. The defaults are the committed campaign settings; you do not need to change anything.
3. *Runtime > Run all*. Allow Drive access when asked.
4. If Colab disconnects, open the notebook again and *Run all* again. Finished steps and finished runs are skipped; an unfinished run starts again from its beginning. Use one Colab session at a time per Drive folder.

**Continue in another Google account.** The results live in the Drive of the account that started the run. To go on from a second account:

1. In the first account, open drive.google.com, right-click the folder `bvp-boundary-pinn`, choose *Share* and add the second account as *Editor*.
2. In the second account, open drive.google.com, then *Shared with me*, right-click `bvp-boundary-pinn`, choose *Organize > Add shortcut* and put the shortcut in *My Drive*.
3. In the second account, open this notebook from GitHub, leave `DRIVE_FOLDER` as it is, pick a GPU runtime and *Run all*.

Step A then lists what it found: the first session's commit and every finished run. Finished runs are never trained again. `MUST_RESUME = True` stops the notebook if the folder holds no earlier session, so a missing shortcut cannot start the campaign over by mistake. If the shortcut does not show up, download the folder in the first account (right-click, *Download*, which gives a zip file), unzip it, and upload the folder to *My Drive* of the second account (*New > Folder upload*) under the same name.

**Using the machine better.** On a GPU two runs train at once (`JOBS`). Each run trains exactly as it would alone: same code, same settings, its own process, one thread, its own seed. `SESSION_HOURS` stops new runs from starting when they are not expected to finish before the session ends, and `DISCONNECT_AT_END` gives the GPU back when the main run stops, so unused time is not counted.

**What it does, in order:** connects Drive and makes the results folder, checks the GPU, clones the repository (the main run checks that its training code is the same as the first session's), installs the exact package versions from `uv.lock` in their own environment, then runs `tools/main_run.py`: tests, the classical reference from scratch (with its error bars), a one-minute dry run of the whole chain on your device, a timing measurement with a run-time estimate, the campaign (4 arms x 5 seeds), and the evaluation (tables, figures, metrics). Logs of every step are in the `logs` folder in Drive.

The full campaign takes many hours (the timing step prints an estimate for your machine). One Colab session will usually not finish it: run it again in a new session until the last step says *All steps finished*.

The 22 physical values, Eqs. 14 to 18 and B1 to B12 are fixed and cannot be set here (AGENTS.md section 3).

In [ ]:
# ============================ SETTINGS: everything the run needs is here ============================
# Where the results go, inside "My Drive". The folder is made if it does not exist. Use a new folder for a
# new campaign; reuse the same folder to resume.
DRIVE_FOLDER = "bvp-boundary-pinn/main-run-P4"
MUST_RESUME = True  # stop if DRIVE_FOLDER holds no earlier session (a missing shortcut must not start over).
                    # Set False only to start a new campaign in a new folder.

# The code. REPO_URL is the repository's own address (git remote "origin").
REPO_URL = "https://github.com/ihmorol/bvp-boundary-pinn.git"
REPO_BRANCH = "impl/campaign-readiness"
REPO_COMMIT = ""   # "" = the branch tip. The main run refuses a commit whose training code differs from the
                   # commit this Drive folder started with (only scheduling, evaluation, tools and docs may differ)

# Hardware.
DEVICE = "auto"    # "auto" (GPU if Colab gives one), "cuda" or "cpu"
JOBS = "auto"      # runs at once: "auto" = 2 on a GPU (free Colab has 2 CPUs), the CPU count (at most 4) on CPU
SESSION_HOURS = 4.5  # free Colab GPU sessions ended after about 5 h here: no run starts unless it is expected to
                     # end within this many hours of the session start. None = no limit (for example a laptop)
DISCONNECT_AT_END = True  # give the runtime back when the main run stops, so idle time is not counted

# What to run. None = what configs/pinn.json says (arms B0 to B3, seeds 0 to 4, campaign "P4").
ARMS = None
SEEDS = None
CAMPAIGN_NAME = None

# Training settings, written into configs/pinn.json ("common" block). These are the committed values.
# Changing one makes new run folders; a Drive folder started with other values refuses to continue.
PINN_SETTINGS = {
    "n_points": 2000,       # training points on [0, 20], clustered at the wall
    "n_monitor": 1000,      # separate monitor points (never trained on)
    "width": 40,            # neurons per hidden layer
    "depth": 4,             # hidden layers
    "adam_steps": 20000,    # Adam steps in total (split over the stages of B2 and B3)
    "lr": 0.001,            # Adam learning rate at the start
    "lr_end": 0.0001,       # and at the end (exponential decay)
    "lbfgs_iters": 5000,    # L-BFGS iterations (log scale)
    "lm_iters": 260,        # Levenberg-Marquardt steps (balanced scale)
    "windows": [0.0, 0.5, 2.0, 6.0, 20.0],  # window edges of B1, B2, B3
}

# Checks around the campaign.
RUN_TESTS = True          # pytest and the record check before anything else
SMOKE_TEST = True         # one-minute dry run of every arm on this device before the real runs
MEASURE_TIMING = True     # measure step times here and print the run-time estimate
CLASSICAL_REPEATS = 5     # repeats of each solve_bvp timing (figure FX1, table TX1)

# Software, exact versions. Every other package comes pinned from uv.lock.
PYTHON_VERSION = "3.12.13"
UV_VERSION = "0.10.9"
TORCH_VERSION = "2.15.0"  # GPU: the PyPI build (CUDA 13.2); CPU: 2.15.0+cpu, as in uv.lock
# ======================================== END OF SETTINGS ========================================

In [ ]:
# Step A: GPU check, Google Drive, results folder.
import json, os, pathlib, shutil, subprocess, sys, time

# When this runtime started: kept on the runtime's own disk, so running the cells again does not move it.
_t0 = pathlib.Path("/content/session_start.txt")
if not _t0.exists():
    _t0.write_text(str(time.time()))
SESSION_START = float(_t0.read_text())

# CPU runtimes have no nvidia-smi at all.
q = subprocess.run(["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"],
                   capture_output=True, text=True) if shutil.which("nvidia-smi") else None
HAS_GPU = q is not None and q.returncode == 0 and bool(q.stdout.strip())
print("GPU (name, driver, memory):", q.stdout.strip() if HAS_GPU else
      "none. For a GPU: Runtime > Change runtime type > GPU, then Run all again.")
if DEVICE == "cuda" and not HAS_GPU:
    raise RuntimeError("DEVICE is 'cuda' but this runtime has no GPU. Pick a GPU runtime or set DEVICE = 'auto'.")
WANT_GPU = HAS_GPU and DEVICE != "cpu"

from google.colab import drive
drive.mount("/content/drive")
OUT = pathlib.Path("/content/drive/MyDrive") / DRIVE_FOLDER
first = OUT / "steps" / "start.json"
if first.exists():
    info = json.loads(first.read_text())
    runs = sorted(p for p in (OUT / "runs").glob("bvp-*") if (p / "meta.json").exists())
    print(f"Resuming. First session: {info.get('started')}, commit {info['commit'][:8]}.")
    print(f"Finished runs found ({len(runs)}), never trained again:", ", ".join(p.name for p in runs) or "none")
    print("Finished steps:", ", ".join(sorted(p.stem for p in (OUT / "steps").glob("*.json"))))
elif MUST_RESUME:
    raise RuntimeError(f"No earlier session in {OUT}. If you meant to continue, the folder is not visible here: in "
                       "another Google account, add the shared folder as a shortcut in My Drive (see the notes at the "
                       "top). To start a new campaign on purpose, set MUST_RESUME = False.")
OUT.mkdir(parents=True, exist_ok=True)
print("Results folder:", OUT)

In [ ]:
# Step B: the repository, at the branch tip or at REPO_COMMIT.
REPO = pathlib.Path("/content/bvp-boundary-pinn")

def token():
    # Only needed if the repository is private: put a GitHub token in Colab's Secrets (key icon) as GITHUB_TOKEN.
    try:
        from google.colab import userdata
        return userdata.get("GITHUB_TOKEN")
    except Exception:
        return None

def git(*args, secret=None, cwd=REPO):
    shown = " ".join(args) if not secret else " ".join(args).replace(secret, "***")
    print("$ git", shown)
    r = subprocess.run(["git", *args], cwd=cwd, capture_output=True, text=True)
    if r.returncode:
        msg = (r.stderr or r.stdout).replace(secret, "***") if secret else (r.stderr or r.stdout)
        raise RuntimeError(f"git {shown} failed: {msg.strip()}")
    return r.stdout.strip()

def fetch_url():
    t = token()
    return (REPO_URL.replace("https://", f"https://x-access-token:{t}@"), t) if t else (REPO_URL, None)

if not (REPO / ".git").exists():
    try:
        git("clone", "--quiet", REPO_URL, str(REPO), cwd="/content")
    except RuntimeError as err:
        url, t = fetch_url()
        if not t:
            raise RuntimeError(f"{err}\nIf the repository is private, add a GitHub token as the Colab secret GITHUB_TOKEN.")
        git("clone", "--quiet", url, str(REPO), secret=t, cwd="/content")
        git("remote", "set-url", "origin", REPO_URL)  # do not keep the token in .git/config

url, t = fetch_url()
git("fetch", "--quiet", url, f"+refs/heads/{REPO_BRANCH}:refs/remotes/origin/{REPO_BRANCH}", secret=t)
# The main run compares this commit's training code with the commit the Drive folder started with, and stops
# if they differ; newer commits may only change scheduling, evaluation, tools, tests and docs.
if REPO_COMMIT:
    print("Using commit", REPO_COMMIT, "(REPO_COMMIT)")
    git("checkout", "--quiet", "--force", REPO_COMMIT)
else:
    git("checkout", "--quiet", "--force", "-B", REPO_BRANCH, f"origin/{REPO_BRANCH}")
git("clean", "-fdq", "--", "configs", "src", "tools", "tests", "engine")
print("Commit:", git("log", "-1", "--format=%H %s"))

In [ ]:
# Step C: Python and the exact packages, in their own environment (/content/venv).
VENV = pathlib.Path("/content/venv")
PY = VENV / "bin" / "python"

# Colab sets these for its own Python 3.13 kernel. Inherited by our Python 3.12 environment they break it:
# MPLBACKEND names an inline backend that is not installed there (matplotlib refuses to import), and
# PYTHONPATH can pull in Colab's own packages. Every command below runs without them.
VENV_ENV = {k: v for k, v in os.environ.items() if k not in ("PYTHONPATH", "PYTHONHOME", "PYTHONSTARTUP", "MPLBACKEND")}
VENV_ENV.update({"MPLBACKEND": "Agg", "PYTHONNOUSERSITE": "1", "PYTHONUNBUFFERED": "1"})

def sh(cmd, cwd=None):
    # Run a command, show its output here (Colab hides output written by subprocesses), stop on failure.
    print("$", " ".join(map(str, cmd)), flush=True)
    proc = subprocess.Popen(list(map(str, cmd)), cwd=cwd, env=VENV_ENV, text=True,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1)
    tail = []
    for line in proc.stdout:
        print(line, end="", flush=True)
        tail = (tail + [line.rstrip()])[-30:]
    if proc.wait():
        raise RuntimeError(f"command failed (exit code {proc.returncode}): {' '.join(map(str, cmd))}\n" + "\n".join(tail))

def venv_torch():
    r = subprocess.run([str(PY), "-c", "import torch; print(torch.__version__, torch.cuda.is_available())"],
                       capture_output=True, text=True, env=VENV_ENV)
    if r.returncode and PY.exists():
        print("PyTorch does not import yet:", (r.stderr or r.stdout).strip().splitlines()[-1:] or "")
    return r.stdout.split() if r.returncode == 0 else None

marker = VENV / "installed.json"
wanted = {"python": PYTHON_VERSION, "torch": TORCH_VERSION, "gpu": WANT_GPU,
          "lock": subprocess.run(["sha256sum", str(REPO / "uv.lock")], capture_output=True, text=True).stdout.split()[0]}
if marker.exists() and json.loads(marker.read_text()).get("wanted") == wanted and venv_torch():
    print("Environment already installed:", venv_torch())
else:
    sh([sys.executable, "-m", "pip", "install", "-q", f"uv=={UV_VERSION}"])
    sh(["uv", "venv", "--clear", "--python", PYTHON_VERSION, str(VENV)])  # uv downloads this Python if needed
    req = subprocess.run(["uv", "export", "--frozen", "--all-groups", "--no-hashes", "--no-emit-project",
                          "--no-header", "--format", "requirements-txt"],
                         cwd=REPO, capture_output=True, text=True, check=True, env=VENV_ENV).stdout
    keep = [l for l in req.splitlines() if l.strip() and not l.lstrip().startswith(("#", "torch=="))]
    (VENV / "requirements.txt").write_text("\n".join(keep) + "\n")
    sh(["uv", "pip", "install", "--python", PY, "-r", VENV / "requirements.txt"])
    if WANT_GPU:
        sh(["uv", "pip", "install", "--python", PY, f"torch=={TORCH_VERSION}"])
        ok = venv_torch()
        if not ok or ok[1] != "True":
            # The CUDA 13.2 build needs a recent NVIDIA driver. If Colab's driver is older, use the PyTorch
            # build Colab itself ships for this GPU (recorded in environment-*.json), so the GPU still works.
            colab = subprocess.run([sys.executable, "-c", "import torch; print(torch.__version__)"],
                                   capture_output=True, text=True).stdout.strip()
            base, _, tag = colab.partition("+")
            print(f"\nWARNING: torch {TORCH_VERSION} cannot use this GPU (driver too old?). "
                  f"Falling back to Colab's own build, torch {colab}.\n")
            index = f"https://download.pytorch.org/whl/{tag}" if tag.startswith("cu") else "https://pypi.org/simple"
            sh(["uv", "pip", "install", "--python", PY, f"torch=={base}", "--index-url", index,
                "--extra-index-url", "https://pypi.org/simple", "--index-strategy", "unsafe-best-match"])
    else:
        sh(["uv", "pip", "install", "--python", PY, "--no-deps", f"torch=={TORCH_VERSION}+cpu",
            "--index-url", "https://download.pytorch.org/whl/cpu"])
    if not venv_torch():
        raise RuntimeError("PyTorch does not import in the new environment; see the messages above.")
    if WANT_GPU and venv_torch()[1] != "True":
        raise RuntimeError("This runtime has a GPU, but no PyTorch build here can use it. Set DEVICE = 'cpu' to run "
                           "on the CPU, or pick another GPU runtime, then Run all again.")
    marker.write_text(json.dumps({"wanted": wanted, "installed": venv_torch()}))
print("torch, GPU usable:", venv_torch())
sh([PY, "-c", "import numpy, scipy, matplotlib, sympy, pytest, sys; "
    "print('python', sys.version.split()[0], 'numpy', numpy.__version__, 'scipy', scipy.__version__, "
    "'matplotlib', matplotlib.__version__, 'sympy', sympy.__version__, 'pytest', pytest.__version__)"])

In [ ]:
# Step D: the main run. Safe to run again after a disconnect: finished steps and runs are skipped.
settings = {"out_dir": str(OUT), "device": DEVICE, "jobs": JOBS, "arms": ARMS, "seeds": SEEDS,
            "campaign_name": CAMPAIGN_NAME, "pinn_common": PINN_SETTINGS, "run_tests": RUN_TESTS,
            "smoke_test": SMOKE_TEST, "measure_timing": MEASURE_TIMING, "classical_repeats": CLASSICAL_REPEATS,
            "session_hours": SESSION_HOURS, "session_start": SESSION_START}
(OUT / "run_settings.json").write_text(json.dumps(settings, indent=2) + "\n")
import signal
proc = subprocess.Popen([str(PY), "tools/main_run.py", "--settings", str(OUT / "run_settings.json")], cwd=REPO,
                        env=VENV_ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                        start_new_session=True)  # VENV_ENV from step C: no Colab PYTHONPATH or MPLBACKEND
try:
    for line in proc.stdout:
        print(line, end="", flush=True)
except KeyboardInterrupt:
    # Stop the whole process tree, so no training keeps running in the background. The run in progress
    # has no meta.json yet, so the next Run all trains it again from its start.
    os.killpg(proc.pid, signal.SIGTERM)
    proc.wait()
    raise
if proc.wait():
    raise RuntimeError(f"The main run stopped with an error (exit code {proc.returncode}). The message above names the "
                     f"step; its full log is in {OUT / 'logs'}. Fix the cause, then Run all again to resume.")

In [ ]:
# Step E: a first look at the results (everything is also in Drive).
from IPython.display import Image, Markdown, display
name = CAMPAIGN_NAME or json.loads((REPO / "configs" / "pinn.json").read_text())["campaign"]["name"]
res = OUT / "results" / name
for f in ("accuracy.md", "wall_values.md", "cost.md", "failures.md"):
    if (res / f).exists():
        display(Markdown((res / f).read_text()))
for f in ("FB1_profiles", "FB2_errors", "FX1_error_vs_time"):
    if (res / "figures" / f"{f}.png").exists():
        display(Image(str(res / "figures" / f"{f}.png"), width=900))
print("All results:", res)

In [ ]:
# Step F: give the runtime back, so the time after the main run is not counted against your Colab limit.
# Everything is in Drive; this page keeps its output.
if DISCONNECT_AT_END:
    print("Disconnecting the runtime now. To continue later: open the notebook and Run all.")
    from google.colab import runtime
    runtime.unassign()
